# Noise2Noise con frames reales y split por escenas

Sin ruido artificial. Las parejas se construyen primero, se agrupan por similitud H-S mediante distancia de Bhattacharyya y luego se dividen por clúster completo para evitar fuga entre entrenamiento, validación y prueba.

In [ ]:
from pathlib import Path
import re
import random
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from tqdm.auto import tqdm
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import pairwise_distances
from skimage.io import imread, imsave
from skimage import img_as_float32
from skimage.color import rgb2gray
from skimage.restoration import estimate_sigma

from careamics import CAREamist
try:
    from careamics.config.factories import create_n2n_config
except ImportError:
    from careamics.config import create_n2n_config

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

: 

In [ ]:
FRAMES_DIR = Path(r"Videos\video30min-11to22\frames")
OUT_DIR = Path(r"Videos\video30min-11to22\n2n")
WORK_DIR = OUT_DIR / "_experimento"
MODEL_DIR = WORK_DIR / "modelos"
REPORT_DIR = WORK_DIR / "reportes"

for directory in (OUT_DIR, WORK_DIR, MODEL_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

PAIR_OFFSET = 1
DISJOINT_PAIRS = True
BHATTACHARYYA_THRESHOLD = 0.45
H_BINS = 32
S_BINS = 32
HISTOGRAM_MAX_SIDE = 384
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
PATCH_SIZE = [256, 256]
BATCH_SIZE = 24
NUM_EPOCHS = 100
TILE_SIZE = [512, 512]
TILE_OVERLAP = [64, 64]
METRIC_SAMPLE_SIZE = 100

assert FRAMES_DIR.exists(), f"No existe la carpeta: {FRAMES_DIR.resolve()}"
assert np.isclose(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 1.0)
print("Entrada:", FRAMES_DIR.resolve())
print("Salida:", OUT_DIR.resolve())

In [ ]:
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def natural_sort_key(path):
    return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", path.name.lower())]

frame_files = sorted(
    [p for p in FRAMES_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS],
    key=natural_sort_key,
)
assert len(frame_files) >= 6, "Se necesitan al menos 6 frames."

sample = img_as_float32(imread(frame_files[0]))
if sample.ndim == 2:
    CAREAMICS_AXES = "YX"
elif sample.ndim == 3:
    CAREAMICS_AXES = "YXC"
else:
    raise ValueError(f"Shape no soportado: {sample.shape}")

print("Frames:", len(frame_files))
print("Shape:", sample.shape)
print("Axes:", CAREAMICS_AXES)

In [ ]:
def build_pair_records(files, offset=1, disjoint=True):
    step = offset + 1 if disjoint else 1
    records = []
    for pair_id, i in enumerate(range(0, len(files) - offset, step)):
        records.append({
            "pair_id": pair_id,
            "source_index": i,
            "target_index": i + offset,
            "source_path": files[i],
            "target_path": files[i + offset],
        })
    return records

pair_records = build_pair_records(frame_files, PAIR_OFFSET, DISJOINT_PAIRS)
assert len(pair_records) >= 3
print("Parejas:", len(pair_records))
for record in pair_records[:5]:
    print(record["source_path"].name, "->", record["target_path"].name)

In [ ]:
def read_rgb_for_histogram(path):
    image = imread(path)
    if image.ndim == 2:
        image = np.stack([image, image, image], axis=-1)
    elif image.ndim == 3 and image.shape[-1] == 1:
        image = np.repeat(image, 3, axis=-1)
    elif image.ndim == 3 and image.shape[-1] >= 4:
        image = image[..., :3]

    image = np.clip(img_as_float32(image), 0, 1)
    height, width = image.shape[:2]
    largest_side = max(height, width)
    if largest_side > HISTOGRAM_MAX_SIDE:
        scale = HISTOGRAM_MAX_SIDE / largest_side
        new_size = (max(1, round(width * scale)), max(1, round(height * scale)))
        image = cv2.resize(image, new_size, interpolation=cv2.INTER_AREA)
    return np.round(image * 255).astype(np.uint8)

def hs_histogram(path):
    rgb = read_rgb_for_histogram(path)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [H_BINS, S_BINS], [0, 180, 0, 256])
    hist = hist.astype(np.float64).ravel() + 1e-12
    return hist / hist.sum()

pair_descriptors = []
for record in tqdm(pair_records, desc="Descriptores H-S"):
    descriptor = 0.5 * hs_histogram(record["source_path"]) + 0.5 * hs_histogram(record["target_path"])
    pair_descriptors.append(descriptor / descriptor.sum())
pair_descriptors = np.stack(pair_descriptors)
print(pair_descriptors.shape)

In [ ]:
def bhattacharyya_distance(a, b):
    coefficient = np.clip(np.sum(np.sqrt(a * b)), 1e-12, 1.0)
    return -np.log(coefficient)

distance_matrix = pairwise_distances(pair_descriptors, metric=bhattacharyya_distance)
np.fill_diagonal(distance_matrix, 0.0)
assert np.all(np.isfinite(distance_matrix))
assert np.allclose(distance_matrix, distance_matrix.T, atol=1e-8)

try:
    clusterer = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=BHATTACHARYYA_THRESHOLD,
    )
except TypeError:
    clusterer = AgglomerativeClustering(
        n_clusters=None,
        affinity="precomputed",
        linkage="average",
        distance_threshold=BHATTACHARYYA_THRESHOLD,
    )

cluster_labels = clusterer.fit_predict(distance_matrix)
for record, label in zip(pair_records, cluster_labels):
    record["cluster_id"] = int(label)

cluster_sizes = pd.Series(cluster_labels).value_counts()
print("Escenas:", len(cluster_sizes))
print(cluster_sizes.describe())

In [ ]:
def split_clusters_by_size(labels, ratios, seed=42):
    cluster_ids, counts = np.unique(labels, return_counts=True)
    if len(cluster_ids) < 3:
        raise RuntimeError("Se detectaron menos de 3 escenas. Pruebe BHATTACHARYYA_THRESHOLD = 0.35")

    rng = np.random.default_rng(seed)
    items = list(zip(cluster_ids.tolist(), counts.tolist()))
    rng.shuffle(items)
    items.sort(key=lambda x: x[1], reverse=True)

    names = ["train", "validation", "test"]
    total = len(labels)
    targets = {name: ratio * total for name, ratio in zip(names, ratios)}
    assigned = {name: [] for name in names}
    current = {name: 0 for name in names}

    for name, (cluster_id, size) in zip(names, items[:3]):
        assigned[name].append(cluster_id)
        current[name] += size

    for cluster_id, size in items[3:]:
        name = min(names, key=lambda n: (current[n] + size - targets[n]) ** 2)
        assigned[name].append(cluster_id)
        current[name] += size
    return assigned, current

split_clusters, split_counts = split_clusters_by_size(
    cluster_labels,
    (TRAIN_RATIO, VAL_RATIO, TEST_RATIO),
    SEED,
)

sets = {name: set(ids) for name, ids in split_clusters.items()}
train_records = [r for r in pair_records if r["cluster_id"] in sets["train"]]
val_records = [r for r in pair_records if r["cluster_id"] in sets["validation"]]
test_records = [r for r in pair_records if r["cluster_id"] in sets["test"]]

print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))

In [ ]:
def used_frames(records):
    result = set()
    for record in records:
        result.add(record["source_path"].resolve())
        result.add(record["target_path"].resolve())
    return result

train_frames = used_frames(train_records)
val_frames = used_frames(val_records)
test_frames = used_frames(test_records)

assert train_frames.isdisjoint(val_frames), "Leakage train-validation"
assert train_frames.isdisjoint(test_frames), "Leakage train-test"
assert val_frames.isdisjoint(test_frames), "Leakage validation-test"
print("Auditoría superada: no hay frames compartidos entre splits.")

manifest = []
for split_name, records in (("train", train_records), ("validation", val_records), ("test", test_records)):
    for r in records:
        manifest.append({
            "split": split_name,
            "cluster_id": r["cluster_id"],
            "pair_id": r["pair_id"],
            "source_file": r["source_path"].name,
            "target_file": r["target_path"].name,
        })
pd.DataFrame(manifest).to_csv(REPORT_DIR / "split_escenas_n2n.csv", index=False)

In [ ]:
expected_shape = imread(pair_records[0]["source_path"]).shape

def load_pairs(records, description):
    sources, targets = [], []
    for record in tqdm(records, desc=description):
        source = img_as_float32(imread(record["source_path"]))
        target = img_as_float32(imread(record["target_path"]))
        if source.shape != expected_shape or target.shape != expected_shape:
            raise ValueError(f"Shape inconsistente en pareja {record['pair_id']}")
        sources.append(source)
        targets.append(target)
    return np.stack(sources).astype(np.float32), np.stack(targets).astype(np.float32)

x_train, y_train = load_pairs(train_records, "Train")
x_val, y_val = load_pairs(val_records, "Validation")
print(x_train.shape, y_train.shape, x_val.shape, y_val.shape)

In [ ]:
config = create_n2n_config(
    experiment_name="video30min_real_n2n",
    data_type="array",
    axes=CAREAMICS_AXES,
    patch_size=PATCH_SIZE,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
)
print(config)

In [ ]:
try:
    careamist = CAREamist(config, work_dir=MODEL_DIR)
except TypeError:
    careamist = CAREamist(config)

careamist.train(
    train_source=x_train,
    train_target=y_train,
    val_source=x_val,
    val_target=y_val,
)

In [ ]:
del x_train, y_train, x_val, y_val
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Memoria liberada")

In [ ]:
def extract_prediction(result):
    predictions = result[0] if isinstance(result, tuple) else result
    prediction = predictions[0] if isinstance(predictions, list) else predictions
    prediction = np.asarray(prediction)
    return prediction

def predict_one(model, image):
    image = img_as_float32(image)
    batched = image[np.newaxis, ...]
    try:
        result = model.predict(source=batched, tile_size=TILE_SIZE, tile_overlap=TILE_OVERLAP)
    except TypeError:
        try:
            result = model.predict(source=batched, tile_size=TILE_SIZE, overlaps=TILE_OVERLAP)
        except TypeError:
            result = model.predict(source=batched)
    prediction = extract_prediction(result)
    while prediction.ndim > image.ndim and prediction.shape[0] == 1:
        prediction = prediction[0]
    return np.clip(prediction.astype(np.float32), 0, 1)

def save_image(path, image):
    image = np.clip(image, 0, 1)
    if path.suffix.lower() in {".tif", ".tiff"}:
        imsave(path, image.astype(np.float32), check_contrast=False)
    else:
        imsave(path, np.round(image * 255).astype(np.uint8), check_contrast=False)

for frame_path in tqdm(frame_files, desc="Inferencia"):
    image = img_as_float32(imread(frame_path))
    prediction = predict_one(careamist, image)
    if prediction.shape != image.shape:
        raise ValueError(f"Shape de salida incorrecto para {frame_path.name}: {prediction.shape} != {image.shape}")
    save_image(OUT_DIR / frame_path.name, prediction)
print("Guardado en:", OUT_DIR.resolve())

In [ ]:
import pyiqa
metric_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
brisque_metric = pyiqa.create_metric("brisque", device=metric_device)
niqe_metric = pyiqa.create_metric("niqe", device=metric_device)

def metric_tensor(image):
    image = np.clip(img_as_float32(image), 0, 1)
    if image.ndim == 2:
        image = np.stack([image] * 3, axis=-1)
    elif image.shape[-1] == 1:
        image = np.repeat(image, 3, axis=-1)
    elif image.shape[-1] >= 4:
        image = image[..., :3]
    return torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1).unsqueeze(0).float().to(metric_device)

def estimated_snr_db(image):
    image = np.clip(img_as_float32(image), 0, 1)
    gray = rgb2gray(image[..., :3]) if image.ndim == 3 else image
    sigma = float(estimate_sigma(gray, channel_axis=None, average_sigmas=True))
    signal_rms = float(np.sqrt(np.mean(gray ** 2)))
    return 20 * np.log10((signal_rms + 1e-12) / (sigma + 1e-12))

@torch.inference_mode()
def evaluate_image(image):
    tensor = metric_tensor(image)
    return {
        "BRISQUE": float(brisque_metric(tensor).item()),
        "NIQE": float(niqe_metric(tensor).item()),
        "SNR": estimated_snr_db(image),
    }

In [ ]:
indices = np.unique(np.linspace(0, len(frame_files) - 1, min(METRIC_SAMPLE_SIZE, len(frame_files)), dtype=int))
rows = []
for index in tqdm(indices, desc="Métricas"):
    original_path = frame_files[index]
    original = img_as_float32(imread(original_path))
    denoised = img_as_float32(imread(OUT_DIR / original_path.name))
    before = evaluate_image(original)
    after = evaluate_image(denoised)
    rows.append({
        "frame": original_path.name,
        "BRISQUE_original": before["BRISQUE"],
        "BRISQUE_n2n": after["BRISQUE"],
        "NIQE_original": before["NIQE"],
        "NIQE_n2n": after["NIQE"],
        "SNR_original": before["SNR"],
        "SNR_n2n": after["SNR"],
    })
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(REPORT_DIR / "metricas_por_frame.csv", index=False)
display(metrics_df.head())

In [ ]:
summary = pd.DataFrame({
    "Original": [
        metrics_df["BRISQUE_original"].mean(),
        metrics_df["NIQE_original"].mean(),
        metrics_df["SNR_original"].mean(),
    ],
    "N2N": [
        metrics_df["BRISQUE_n2n"].mean(),
        metrics_df["NIQE_n2n"].mean(),
        metrics_df["SNR_n2n"].mean(),
    ],
}, index=["BRISQUE ↓", "NIQE ↓", "SNR estimado ↑"])

summary["Mejoró"] = [
    summary.loc["BRISQUE ↓", "N2N"] < summary.loc["BRISQUE ↓", "Original"],
    summary.loc["NIQE ↓", "N2N"] < summary.loc["NIQE ↓", "Original"],
    summary.loc["SNR estimado ↑", "N2N"] > summary.loc["SNR estimado ↑", "Original"],
]
summary.to_csv(REPORT_DIR / "resumen_metricas.csv")
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
plot_info = [
    ("BRISQUE", "BRISQUE ↓", "↓ menor es mejor"),
    ("NIQE", "NIQE ↓", "↓ menor es mejor"),
    ("SNR estimado", "SNR estimado ↑", "↑ mayor es mejor"),
]
for axis, (title, row, direction) in zip(axes, plot_info):
    values = [summary.loc[row, "Original"], summary.loc[row, "N2N"]]
    bars = axis.bar(["Original", "N2N"], values, color=["#7f8c8d", "#2e86de"])
    axis.set_title(f"{title}\n{direction}")
    axis.grid(axis="y", alpha=0.25)
    for bar, value in zip(bars, values):
        axis.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{value:.3f}", ha="center", va="bottom")
    status = "✓ Mejoró" if summary.loc[row, "Mejoró"] else "✗ No mejoró"
    color = "green" if summary.loc[row, "Mejoró"] else "red"
    axis.text(0.5, 0.95, status, transform=axis.transAxes, ha="center", va="top", color=color, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORT_DIR / "comparacion_metricas.png", dpi=200, bbox_inches="tight")
plt.show()